#### 2. LangSmith Datasets

In [19]:
from pathlib import Path
from dotenv import load_dotenv
import os


def load_project_env():
    """Load .env.local from repo root (works regardless of notebook cwd)."""
    for root in [Path.cwd(), *Path.cwd().parents]:
        env_file = root / ".env.local"
        if env_file.is_file():
            load_dotenv(override=True, dotenv_path=env_file)
            # LangSmith Client accepts either env var name
            if os.getenv("LANGCHAIN_API_KEY") and not os.getenv("LANGSMITH_API_KEY"):
                os.environ["LANGSMITH_API_KEY"] = os.environ["LANGCHAIN_API_KEY"]
            return env_file
    raise FileNotFoundError(
        "Could not find .env.local. Add it at the repo root: sweta-ai-labs/.env.local"
    )


env_file = load_project_env()
print(f"Loaded env from: {env_file}")

Loaded env from: /Users/suneelsharma/Documents/sweta/sweta-ai-labs/.env.local


In [20]:
import os
from openai import OpenAI
from langsmith import Client

# --- Setup (reload env so this cell is safe to run alone after kernel restart) ---
env_file = load_project_env()
my_api_key = os.getenv("OPENAI_API_KEY")
langsmith_api_key = os.getenv("LANGCHAIN_API_KEY")

if not langsmith_api_key:
    raise ValueError(
        f"LANGCHAIN_API_KEY missing after loading {env_file}. "
        "Add a LangSmith API key (lsv2_pt_...) to .env.local."
    )
if not langsmith_api_key.startswith("lsv2_"):
    raise ValueError("LANGCHAIN_API_KEY should start with lsv2_ (regenerate at smith.langchain.com/settings).")

# Quick auth check before dataset operations
Client(api_key=langsmith_api_key).list_datasets(limit=1)

print("LangSmith key prefix:", langsmith_api_key[:8])
print("Project:", os.getenv("LANGCHAIN_PROJECT"))
print("Tracing:", os.getenv("LANGCHAIN_TRACING_V2"))
print("Auth OK")

LangSmith key prefix: lsv2_pt_
Project: Langsmith_Eval_June_2026
Tracing: true
Auth OK


In [21]:
from langsmith import Client

load_project_env()
client = Client(api_key=os.environ["LANGCHAIN_API_KEY"])

dataset_name = "Test-Dataset"
if client.has_dataset(dataset_name=dataset_name):
    dataset = client.read_dataset(dataset_name=dataset_name)
    print(f"Using existing dataset: {dataset_name}")
else:
    dataset = client.create_dataset(dataset_name=dataset_name)
    print(f"Created dataset: {dataset_name}")

dataset

Using existing dataset: Test-Dataset


Dataset(name='Test-Dataset', description=None, data_type=<DataType.kv: 'kv'>, id=UUID('7fd8cfb8-25e3-47b3-b25d-6ca498c18f75'), created_at=datetime.datetime(2026, 6, 18, 3, 42, 27, 622839, tzinfo=datetime.timezone.utc), modified_at=datetime.datetime(2026, 6, 18, 3, 42, 27, 622839, tzinfo=datetime.timezone.utc), example_count=0, session_count=0, last_session_start_time=None, inputs_schema=None, outputs_schema=None, transformations=None)

In [22]:
examples = [
{"inputs": {"question": "When is John's pay processed?"}, "outputs": {"answer": "John's pay is processed on the 1st of every month."}},
{"inputs": {"question": "What is Julie's job title?"}, "outputs": {"answer": "Julie is a software engineer."}},
]